# EEPM3: Notebook C - Offline Trainer (FAST / GPU)

**Instructions:**
- Set **Accelerator: GPU T4 x2 (or P100)**.
- Go to "Add Data" > "Your Datasets" and mount the database exported from Notebook B to `../input/edm3-experience-buffer/`.
- The weights will be saved to `/kaggle/working/edm3_v2_weights_final.npz`.

In [ ]:
import os, sys
import subprocess

print("Installing JAX with GPU support...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jax[cuda12]", "flax", "optax"], check=True)
print("Dependencies installed.")

In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import flax.linen as nn
import optax
import sqlite3

# ── Global Config ──────────────────────────────────────────────
SEQ_LEN         = 100_000
VOCAB_SIZE      = 5
NUM_EDITS       = 10
BATCH_SIZE      = 32
LEARNING_RATE   = 1e-4
MAX_GRAD_NORM   = 1.0
ALPHA_GFN       = 0.5
TRAIN_EPOCHS    = 200

# Paths
INPUT_DB_PATH = "../input/edm3-experience-buffer/experience_replay.db"
WORKING_DB_PATH = "/kaggle/working/experience_replay.db"
CKPT_PATH = "/kaggle/working/edm3_v2_weights_final.npz"

print(f"JAX devices: {jax.devices()}")

In [ ]:
# Copy DB to /kaggle/working/ so we can augment it without read-only errors on the mounted dataset
if os.path.exists(INPUT_DB_PATH):
    print(f"Copying dataset {INPUT_DB_PATH} to working directory for RBS Augmentation...")
    import shutil
    shutil.copyfile(INPUT_DB_PATH, WORKING_DB_PATH)
else:
    raise FileNotFoundError(f"{INPUT_DB_PATH} not found! Did you mount the dataset from Notebook B?")

In [ ]:
def rbs_augment(db_path, top_pct=0.10, hallucinations_per=5):
    """Retrospective Backward Synthesis: multiply high-reward signal."""
    conn = sqlite3.connect(db_path)
    rows = conn.execute(
        "SELECT trajectory_id, actions, forward_log_probs, reward "
        "FROM experiences ORDER BY reward DESC"
    ).fetchall()

    if not rows:
        print("No experiences to augment!")
        return

    total = len(rows)
    cutoff = max(1, int(total * top_pct))
    top_rows = rows[:cutoff]
    print(f"RBS: Augmenting top {cutoff}/{total} experiences (reward >= {top_rows[-1][4]:.6f})")

    rng = np.random.default_rng(42)
    augmented = 0

    for traj_id, act_bytes, lp_bytes, reward in top_rows:
        actions = np.frombuffer(act_bytes, dtype=np.int32).copy()
        mutations = [(int(a) // VOCAB_SIZE, int(a) % VOCAB_SIZE) for a in actions if a != SEQ_LEN * VOCAB_SIZE]

        if len(mutations) < 2: continue
        seen = set()

        for _ in range(hallucinations_per * 3):
            if len(seen) >= hallucinations_per: break
            perm = rng.permutation(len(mutations))
            key = tuple(perm)
            if key in seen: continue
            seen.add(key)

            perm_actions = np.array([
                mutations[i][0] * VOCAB_SIZE + mutations[i][1] for i in perm
            ], dtype=np.int32)

            if len(perm_actions) < NUM_EDITS:
                pad = np.full(NUM_EDITS - len(perm_actions), SEQ_LEN * VOCAB_SIZE, dtype=np.int32)
                perm_actions = np.concatenate([perm_actions, pad])

            syn_lp = np.array([-np.log(max(SEQ_LEN - i, 1) * 4) for i in range(NUM_EDITS)], dtype=np.float32)

            conn.execute(
                "INSERT INTO experiences (trajectory_id, actions, forward_log_probs, reward) "
                "VALUES (?, ?, ?, ?)",
                (total + augmented, perm_actions.tobytes(), syn_lp.tobytes(), reward),
            )
            augmented += 1

    conn.commit()
    final = conn.execute("SELECT COUNT(*) FROM experiences").fetchone()[0]
    conn.close()
    print(f"RBS complete: {augmented} hallucinated trajectories added")
    print(f"Total experiences: {final} ({final/total:.1f}x original)")

rbs_augment(WORKING_DB_PATH)

In [ ]:
class ConvergenceTracker:
    def __init__(self, alpha=0.95, threshold_pct=0.05, window=50, var_thresh=0.01):
        self.alpha = alpha
        self.threshold_pct = threshold_pct
        self.window = window
        self.var_thresh = var_thresh
        self.ema = None
        self.baseline_ema = None
        self.losses = []
        self.converged = False
        self.convergence_epoch = None

    def update(self, loss, epoch):
        self.losses.append(loss)
        if self.ema is None:
            self.ema = loss
            self.baseline_ema = loss
        else:
            self.ema = self.alpha * self.ema + (1 - self.alpha) * loss

        if len(self.losses) >= self.window and not self.converged:
            pct_drop = (self.baseline_ema - self.ema) / max(abs(self.baseline_ema), 1e-8)
            recent_var = np.var(self.losses[-self.window:])
            if pct_drop > self.threshold_pct and recent_var < self.var_thresh:
                self.converged = True
                self.convergence_epoch = epoch
        return self.converged

def load_replay_data(db_path, batch_size=BATCH_SIZE):
    conn = sqlite3.connect(db_path)
    rows = conn.execute("SELECT actions, forward_log_probs, reward FROM experiences").fetchall()
    conn.close()

    if not rows:
        raise ValueError("No experiences in replay buffer!")

    all_actions = [np.frombuffer(r[0], dtype=np.int32).copy() for r in rows]
    all_lp = [np.frombuffer(r[1], dtype=np.float32).copy() for r in rows]
    all_rewards = [float(r[2]) for r in rows]

    print(f"Loaded {len(rows)} experiences")
    rewards = np.array(all_rewards)
    print(f"  Reward: mean={rewards.mean():.4f}, std={rewards.std():.4f}, "
          f"min={rewards.min():.4f}, max={rewards.max():.4f}")
    return all_actions, all_lp, all_rewards

def iter_batches(all_actions, all_lp, all_rewards, batch_size, rng_key=None):
    n = len(all_rewards)
    indices = np.arange(n)
    if rng_key is not None:
        np.random.seed(int(jax.random.randint(rng_key, (), 0, 2**31)))
        np.random.shuffle(indices)

    for b in range(n // batch_size):
        idx = indices[b * batch_size : (b + 1) * batch_size]
        yield {
            "forward_log_probs": jnp.array(np.stack([all_lp[i] for i in idx])),
            "rewards": jnp.array([all_rewards[i] for i in idx]),
        }

In [ ]:
def alpha_gfn_tb_loss(log_z, forward_log_probs, log_reward, alpha, num_edits):
    sum_log_pf = jnp.sum(forward_log_probs[:num_edits])
    log_pb_terms = jnp.array([-jnp.log(jnp.float32(num_edits - t)) for t in range(num_edits)])
    sum_log_pb = jnp.sum(log_pb_terms)
    residual = log_z + alpha * sum_log_pf - log_reward - (1.0 - alpha) * sum_log_pb
    return residual ** 2

optimizer = optax.chain(
    optax.clip_by_global_norm(MAX_GRAD_NORM),
    optax.adamw(learning_rate=LEARNING_RATE),
)

@jax.jit
def train_step(log_z, opt_state, batch_lp, batch_rewards):
    def loss_fn(lz):
        log_r = jnp.log(jnp.maximum(batch_rewards, 1e-8))
        losses = jax.vmap(
            lambda lp, lr: alpha_gfn_tb_loss(lz, lp, lr, ALPHA_GFN, NUM_EDITS)
        )(batch_lp, log_r)
        return jnp.mean(losses)

    loss, grad = jax.value_and_grad(loss_fn)(log_z)
    updates, new_opt_state = optimizer.update(grad, opt_state, log_z)
    new_log_z = optax.apply_updates(log_z, updates)
    grad_norm = jnp.sqrt(grad ** 2)
    return loss, new_log_z, new_opt_state, grad_norm

In [ ]:
all_act, all_lp, all_rew = load_replay_data(WORKING_DB_PATH)

log_z = jnp.float32(0.0)
opt_state = optimizer.init(log_z)
tracker = ConvergenceTracker()

print(f"\n{'Epoch':>6} | {'Mean Loss':>12} | {'EMA':>10} | {'log_Z':>10} | {'Grad':>10}")
print("-" * 62)

key = jax.random.PRNGKey(9999)
for epoch in range(1, TRAIN_EPOCHS + 1):
    key, ek = jax.random.split(key)
    losses, grads = [], []

    for batch in iter_batches(all_act, all_lp, all_rew, BATCH_SIZE, ek):
        loss, log_z, opt_state, gn = train_step(log_z, opt_state, batch["forward_log_probs"], batch["rewards"])
        losses.append(float(loss))
        grads.append(float(gn))

    ml = np.mean(losses)
    mg = np.mean(grads)
    converged = tracker.update(ml, epoch)

    if epoch == 1 or epoch % 10 == 0 or converged:
        print(f"{epoch:>6} | {ml:>12.4f} | {tracker.ema:>10.4f} | {float(log_z):>10.6f} | {mg:>10.6f}")

    if converged and tracker.convergence_epoch == epoch:
        print(f"\n*** CONVERGENCE at epoch {epoch} ***")

print("-" * 62)
pct = (tracker.baseline_ema - tracker.ema) / max(abs(tracker.baseline_ema), 1e-8) * 100
if tracker.converged:
    print(f"✅ CONVERGENCE VALIDATED at epoch {tracker.convergence_epoch} ({pct:.2f}% EMA drop)")
else:
    print(f"❌ Not converged in {TRAIN_EPOCHS} epochs ({pct:.2f}% EMA drop, threshold 5%)")

np.savez(CKPT_PATH, log_z=np.array(float(log_z)))
print(f"\nCheckpoint saved: {CKPT_PATH}")
print(f"Final log_z: {float(log_z):.6f}")